<a href="https://colab.research.google.com/github/HubbaBubba64/AI-Grammar-Score-Prototype/blob/main/week2_classification/grammar_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets torch scikit-learn pandas

In [2]:
import pandas as pd
url = "https://raw.githubusercontent.com/HubbaBubba64/AI-Grammar-Score-Prototype/main/data/grammar_sample.csv"

df = pd.read_csv(url)


print(df.head())

                                                text label
0       I have been studying in Japan for two years.  good
1                         I studying Japan two year.   bad
2  The experiment was successful with three samples.  good
3                Experiment successful three sample.   bad
4       Please contact me if you have any questions.  good


In [3]:
label_map = {
    "bad": 0,
    "good": 1
}

df["labels"] = df["label"].map(label_map)

print(df.head())


                                                text label  labels
0       I have been studying in Japan for two years.  good       1
1                         I studying Japan two year.   bad       0
2  The experiment was successful with three samples.  good       1
3                Experiment successful three sample.   bad       0
4       Please contact me if you have any questions.  good       1


In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [6]:
encodings = tokenizer(
    list(df["text"]),
    truncation=True,
    padding=True,
    return_tensors="pt"
)

print(encodings.keys())


KeysView({'input_ids': tensor([[  101,  1045,  2031,  2042,  5702,  1999,  2900,  2005,  2048,  2086,
          1012,   102,     0],
        [  101,  1045,  5702,  2900,  2048,  2095,  1012,   102,     0,     0,
             0,     0,     0],
        [  101,  1996,  7551,  2001,  3144,  2007,  2093,  8168,  1012,   102,
             0,     0,     0],
        [  101,  7551,  3144,  2093,  7099,  1012,   102,     0,     0,     0,
             0,     0,     0],
        [  101,  3531,  3967,  2033,  2065,  2017,  2031,  2151,  3980,  1012,
           102,     0,     0],
        [  101,  3967,  2033,  2065,  2031,  3160,  1012,   102,     0,     0,
             0,     0,     0],
        [  101,  2057, 16578,  1996,  3463,  2478,  1037,  7778,  2944,  1012,
           102,     0,     0],
        [  101,  2057, 17908,  2765,  2478,  6747,  2944,  1012,   102,     0,
             0,     0,     0],
        [  101,  1996,  3034,  2097,  2202,  2173,  2279,  5958,  1012,   102,
             0,   

In [7]:
import torch

labels = torch.tensor(df["labels"].values)

print(labels)

tensor([1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0,
        1, 0, 1, 0, 1, 0])


In [8]:
from torch.utils.data import Dataset

class GrammarDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {
            key: val[idx]
            for key, val in self.encodings.items()
        }
        item["labels"] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

dataset = GrammarDataset(encodings, labels)

In [9]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

Train size: 24
Test size: 6


In [10]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    logging_dir="./logs",
    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="no"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [12]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [13]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.537858,0.719952
2,0.398291,0.579847
3,0.071248,0.724769
4,0.194047,0.766084
5,0.087506,0.691413


TrainOutput(global_step=30, training_loss=0.33346511696775755, metrics={'train_runtime': 40.5115, 'train_samples_per_second': 2.962, 'train_steps_per_second': 0.741, 'total_flos': 801666496800.0, 'train_loss': 0.33346511696775755, 'epoch': 5.0})

In [14]:
trainer.evaluate()

{'eval_loss': 0.6914132237434387,
 'eval_runtime': 0.4884,
 'eval_samples_per_second': 12.286,
 'eval_steps_per_second': 4.095,
 'epoch': 5.0}

## Week 2 Day 2 Notes

- I trained my first BERT-based binary classifier.
- The model learns to classify sentences as good or bad.
- The dataset is very small, so accuracy is not the main goal yet.
- The main goal is to understand the training pipeline.